## Testing the association between covariate embeddings and clinical covariates

In [ ]:
# !bash -c 'eval "$(conda shell.bash hook)" \
#     && conda activate /work/islet_cartography_scrna/scrna_cartography_py_analysis \
#     && python -m ipykernel install --user \
#        --name scrna_cartography_py_analysis \
#        --display-name "py_analysis"'

## Load libraries

In [1]:
import os
import sys
import glob
from pyhere import here

import anndata
import numpy as np
import pandas as pd
import scanpy as sc
from joblib import parallel_backend
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import warnings
import statsmodels.api as sm
from pathlib import Path

import zstandard as zstd
import hdf5plugin

import pickle

# My modules / functions
sys.path.append(str(here('scripts/misc')))  
import my_anndata as ma

IndentationError: expected an indented block after function definition on line 545 (my_anndata.py, line 555)

In [ ]:
test = scPoli.load('/work/islet_cartography_scrna/data/integration/first_pass/models/scpoli_model_1000_technical_integration_ic_id_donor_integrate_latent15_embed10')

In [ ]:
emb = test.get_conditional_embeddings()['ic_id_donor_integrate']
df = pd.DataFrame(
   emb.X,
    index=emb.obs_names,   # cells / samples
    columns=emb.var_names  # embedding dimensions
)

In [ ]:
df

In [ ]:
str(here('/data/integration/first_pass/models/scpoli_model_1000_technical_integration_ic_id_donor_integrate_latent15_embed10/'))

## Parameters

In [ ]:
first_pass = here('data/integration/first_pass/models')

In [ ]:
# Paths
py_analysis = str(here('crna_cartography_py_analysis'))
scpoli = str(here('scrna_cartography_scpoli'))
save_path = str(here('data/integration/first_pass/tmp'))

In [ ]:
from pathlib import Path

## Load embeddings

In [ ]:
embedding_paths = glob.glob(str(first_pass / '*'))

In [ ]:
model_path = glob.glob(str(first_pass / '*'))

In [ ]:
Path(model_path[0]).stem

In [ ]:
import os
def get_covariate_embedding(model_path, save_path, scpoli, uuik_path, adata_path, embed_name="ic_id_donor_integrate"):
    methods = ['scvi', 'sysvi', 'scpoli']
    
    model_path = Path(model_path)
    filename = model_path.stem

    method = next((m for m in methods if m in filename), None)
    base_name = filename

    if method == 'scpoli':
        print(f'Loading ScPoli')
        result_scpoli = subprocess.run([
            "conda", "run", "-p", scpoli, "python", "load_covariate_embedding_scpoli.py",
            str(model_path), str(save_path), embed_name, base_name
        ],capture_output=True, text=True)
        
        covariate = pd.read_csv(os.path.join(save_path, f"{base_name}.csv"), index_col=0)
    
    if method == 'scvi':
        model_number = int((re.search(r"model_(\d+)", base_name) or [1]).group(1))
        
        adata_dir = os.path.join(adata_path, f'adata_{model_number}_hvg.h5ad')

        ad = sc.read_h5ad(adata_dir)

        uuid_dir = os.path.join(uuid_path, f'{base_name}_model_uuid.pkl')
        with open(uuid_dir, "rb") as f:
            meta = pickle.load(f)
        
        for key, value in meta.items():
            if value is not None:
                adata.uns[key] = value

        model = scvi.model.SCVI.load()
        
        return covariate

In [ ]:
import re
base_name = Path(model_path[0]).stem
model_number = int((re.search(r"model_(\d+)", base_name) or [1]).group(1))
model_number

In [ ]:
get_covariate_embedding(model_path = model_path[3], save_path=save_path, scpoli = scpoli)

In [2]:
import scvi
from scvi.external import SysVI
import pickle

In [3]:
ad = sc.read_h5ad('/work/islet_cartography_scrna/data/integration/first_pass/objects/adata_1000_hvg.h5ad')

In [4]:
with open('/work/islet_cartography_scrna/data/integration/first_pass/embedding/sysvi_1000_technical_integration_ic_id_donor_integrate_latent15_embed10_model_uuid.pkl', "rb") as f:
    meta = pickle.load(f)

In [6]:
for key, value in meta.items():
    if value is not None:
        ad.uns[key] = value

In [9]:
model = SysVI.load('/work/islet_cartography_scrna/data/integration/first_pass/models/sysvi_model_1000_technical_integration_ic_id_donor_integrate_latent15_embed10', adata = ad)

INFO     File                                                                                                      
         /work/islet_cartography_scrna/data/integration/first_pass/models/sysvi_model_1000_technical_integration_ic
         _id_donor_integrate_latent15_embed10/model.pt already downloaded                                          
INFO     The model has been initialized                                                                            


In [22]:
embedding_tensor = model.embedding

AttributeError: 'SysVI' object has no attribute 'embedding_tensor'

In [27]:
model.module.embeddings_dict['cov0']

Embedding(270, 10)

In [28]:
import pandas as pd

cov_key = 'cov0'

# Get embedding tensor safely
embedding_tensor = model.module.embeddings_dict['cov0'].weight.detach().cpu().numpy()

# Get row names from AnnData
row_names = ad.obs['ic_id_donor_integrate'].cat.categories

# Make DataFrame
embedding_df = pd.DataFrame(embedding_tensor, index=row_names)

print(embedding_df.shape)
print(embedding_df.head())


(270, 10)
               0         1         2         3         4         5         6  \
ic_1_1 -0.402932 -0.536746 -0.034832 -0.580845  0.065069 -1.898415 -1.410683   
ic_1_2 -0.353076  0.735725  0.559074  0.417938 -0.831660 -1.131951  0.001770   
ic_1_3 -1.021631  1.428820 -0.167928  0.207657 -0.241627  0.125360  0.854604   
ic_1_4  0.425447  0.939633 -0.864725  0.057299  0.493619  0.120595  0.611576   
ic_1_5 -0.609822  0.336732  0.100511  1.524914 -1.281067 -1.464601  0.838886   

               7         8         9  
ic_1_1  0.765963  0.252745 -0.358917  
ic_1_2  1.331748  0.562163 -2.002505  
ic_1_3  1.220287  0.483377 -1.298829  
ic_1_4  0.220727 -0.430415 -0.940035  
ic_1_5  0.248761  0.712705 -0.329693  


In [ ]:
X = sm.add_constant(latent_embeddings[:, 0])  # just one latent dimension
y = clinical_data['hba1c']

model = sm.OLS(y, X).fit()
print(model.summary())